# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets with @id
print('Available Record Sets:')
for rs in dataset.metadata.recordSets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    print('  Fields:')
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'dataType', 'unknown')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record set ids
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if len(record_set_ids) > 0:
    first_id = record_set_ids[0]
    print(f"Columns of record set {first_id}:")
    print(dataframes[first_id].columns.tolist())
    dataframes[first_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Choose a record set and a numeric field for analysis
# You may need to adapt these IDs based on actual available fields
from IPython.display import display

# Use the first record set as example
if len(record_set_ids) > 0:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Find a numeric field (e.g. 'cr:field/coverage', 'cr:field/mw_calculated', or similar)
    numeric_field_id = None
    for rs in dataset.metadata.recordSets:
        if rs.id == rs_id:
            for field in rs.fields:
                if getattr(field, 'dataType', None) in ['schema:Float', 'schema:Integer', 'schema:Number']:
                    numeric_field_id = field.id
                    break
            break

    if numeric_field_id is not None and numeric_field_id in df.columns:
        print(f"Analyzing field: {numeric_field_id}")
        threshold = 10
        numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
        filtered_df = df[numeric_series > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series[filtered_df.index] - numeric_series.mean()) / numeric_series.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group field (e.g. 'cr:field/sample' or similar type Text field)
        group_field_id = None
        for rs in dataset.metadata.recordSets:
            if rs.id == rs_id:
                for field in rs.fields:
                    if getattr(field, 'dataType', None) == 'schema:Text' and field.id in df.columns and field.id != numeric_field_id:
                        group_field_id = field.id
                        break
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.reset_index().head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found in the first record set.")
else:
    print("No record sets found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# If a numeric field was analyzed, plot its distribution
if len(record_set_ids) > 0 and 'numeric_field_id' in locals() and numeric_field_id is not None and numeric_field_id in df.columns:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8, 5))
    numeric_series.hist(bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we successfully loaded and explored the mass spectrometry dataset of human extracellular vesicles. Using the `mlcroissant` library, we:
- Loaded metadata and listed record sets with their field `@id`s.
- Extracted records from croissant dataset into Pandas DataFrames using Croissant entity `@id`s.
- Performed basic exploratory data analysis, including filtering and normalization on a numeric field and examined the distribution.
- Demonstrated how to group data by text/categorical field and visualize field distributions.

This workflow can be adapted to other Croissant-compliant datasets and extended for more advanced use cases.